# Visualizing orbital swath data

Visualizing L2 and L3 data products that are generated from sensors that follow orbital tracks around the earth requires some special care to ensure that the underlying data are combined in reasonable ways. The TiTiler-CMR API surfaces several query parameters from the CMR API that make it possible to filter granules down by attributes such as `TRACK_NUMBER` or other properties that define logical groups of granules.

This notebook contains visualization examples that use granule filters to render coherent views of L2 collections like [NISAR Beta Geocoded Polarimetric Covariance (GCOV)](https://www.earthdata.nasa.gov/data/catalog/asf-nisar-l2-gcov-beta-v1-1) and L3 collections like [OPERA Surface Displacement from Sentinel-1](https://www.earthdata.nasa.gov/es/data/catalog/asf-opera-l3-disp-s1-v1-1).

In [ ]:
import json
from datetime import datetime, timedelta, UTC

import httpx
from folium import GeoJson, GeoJsonPopup, GeoJsonTooltip, LayerControl, Map, TileLayer

# titiler_endpoint = "https://staging.openveda.cloud/api/titiler-cmr"  # staging endpoint
titiler_endpoint = (
    "https://v4jec6i5c0.execute-api.us-west-2.amazonaws.com"  # dev endpoint
)

## NISAR GCOV in India

Shortly before this notebook was written (March 2026) ASF published ~22k granules from the NISAR Beta GCOV collection. It is early days for NISAR but TiTiler-CMR can be used to visualize a mosaic of granules.

The `/bbox/.../assets` endpoint can be used to retrieve the granule metadata that matches a TiTiler-CMR query as a GeoJSON feature collection. This is useful because it allows you to look directly at the metadata from the granules that TiTiler-CMR will use for a set of CMR search terms. 

In [ ]:
india_bbox = (66.191, 6.930, 91.297, 34.657)

nisar_assets_req = httpx.get(
    f"{titiler_endpoint}/bbox/{','.join(str(b) for b in india_bbox)}/assets",
    params={
        "collection_concept_id": "C3622214170-ASF",
        "temporal": "2026-01-01T00:00:00Z/2026-02-01T00:00:00Z",  # January 2026
        "f": "geojson",  # return granules as a GeoJSON FeatureCollection
        "sortkey": "-start_date",  # sort granules in descending order by time
        "skipcovered": True,  # do not return multiple granules with identical geometries
        "items_limit": 400,  # global granule query limit
    },
    timeout=None,
)

nisar_assets_req.raise_for_status()
nisar_assets_geojson = nisar_assets_req.json()

print("found", len(nisar_assets_geojson["features"]), "granules")

In [ ]:
for feature in nisar_assets_geojson["features"]:
    # move additional_attributes into properties so we can use
    # them more easily in maps
    feature["properties"].update(
        {
            attribute["name"]: (
                attribute["values"][0]
                if len(attribute["values"]) == 1
                else attribute["values"]
            )
            for attribute in feature["properties"]["additional_attributes"]
        }
    )

    # drop additional_attributes after moving values out
    feature["properties"].pop("additional_attributes")

    feature["properties"]["datetime"] = datetime.fromisoformat(
        feature["properties"]["temporal_extent"]["range_date_time"][
            "beginning_date_time"
        ]
    ).isoformat()

In [ ]:
m = Map(
    location=((india_bbox[1] + india_bbox[3]) / 2, (india_bbox[0] + india_bbox[2]) / 2),
    zoom_start=5,
)

GeoJson(
    nisar_assets_geojson,
    name="granule footprints",
    tooltip=GeoJsonTooltip(fields=["granule_ur"], localize=True),
    popup=GeoJsonPopup(fields=["datetime", "ASCENDING_DESCENDING", "TRACK_NUMBER"]),
    style_function=lambda x: {
        "color": "magenta"
        if x["properties"]["ASCENDING_DESCENDING"] == "ASCENDING"
        else "blue",
        "weight": 1,
        "fillOpacity": 0,
    },
).add_to(m)

LayerControl().add_to(m)

m

In [ ]:
nisar_tilejson_params = {
    "collection_concept_id": "C3622214170-ASF",
    "temporal": "2026-01-01T00:00:00Z/2026-02-01T00:00:00Z",
    "bounding_box": ",".join(str(b) for b in india_bbox),  # optional
    "sortkey": "-start_date",  # prioritize most recent granules during rendering
    "skipcovered": True,  # skip granules with geometry identical to collected granules
    "exitwhenfull": True,  # exit CMR search once tile geometry is covered by assets
    "variables": ["HHHH", "HVHV"],  # two variables to be combined via expression
    "expression": "10 * log10(b1); 10 * log10(b2); 10 * log10(b1/b2)",
    "rescale": [[-20, 0], [-30, 5], [2, 18]],
}

nisar_frequencyA_params = {
    "group": "/science/LSAR/GCOV/grids/frequencyA",
    "minzoom": 10,
}

nisar_frequencyB_params = {
    "group": "/science/LSAR/GCOV/grids/frequencyB",
    "minzoom": 6,
    "maxzoom": 10,
}

### Visualize a single track

The NISAR granules can be overwhelming for the TiTiler-CMR application. One way to limit the number of granules that any single tile request attempts to open is to apply an orbit/track filter to the CMR granule queries. This is done by specifying an `attribute` query parameter ([docs](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#g-additional-attribute)).

The NISAR granules contain HHHH and VVVV arrays at two resolutions, represented by `frequencyA` (high res) and `frequencyB` (low res). We can use this feature to compose a set of tile layers that will create a visually seamless view of the data by rendering tiles for `frequencyB` at lower zoom levels (zoomed out) and `frequencyA` at high zoom levels (zoomed in).

In [ ]:
# pick a track number then visualize the track
track_number = 149

track_params = {"attribute": f"int,TRACK_NUMBER,{track_number}"}

nisar_track_frequencyA_tilejson_req = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params={
        **nisar_tilejson_params,
        **nisar_frequencyA_params,
        **track_params,
    },
    timeout=None,
)

nisar_track_frequencyA_tilejson_req.raise_for_status()
nisar_track_frequencyA_tilejson = nisar_track_frequencyA_tilejson_req.json()

nisar_track_frequencyB_tilejson_req = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params={
        **nisar_tilejson_params,
        **nisar_frequencyB_params,
        **track_params,
    },
    timeout=None,
)

nisar_track_frequencyB_tilejson_req.raise_for_status()
nisar_track_frequencyB_tilejson = nisar_track_frequencyB_tilejson_req.json()

In [ ]:
m = Map(
    location=((india_bbox[1] + india_bbox[3]) / 2, (india_bbox[0] + india_bbox[2]) / 2),
    zoom_start=7,
)

nisar_track_features = {
    "type": "FeatureCollection",
    "features": [
        feature
        for feature in nisar_assets_geojson["features"]
        if int(feature["properties"]["TRACK_NUMBER"]) == track_number
    ],
}

GeoJson(
    nisar_track_features,
    name="granule footprints",
    popup=GeoJsonPopup(
        fields=["granule_ur", "datetime", "ASCENDING_DESCENDING", "TRACK_NUMBER"]
    ),
    style_function=lambda x: {
        "color": "orange",
        "weight": 2,
        "fillOpacity": 0,
    },
).add_to(m)

TileLayer(
    tiles=nisar_track_frequencyB_tilejson["tiles"][0],
    name="NISAR GCOV frequencyB (lower res)",
    opacity=1,
    attr="NASA/ISRO",
    overlay=True,
    min_zoom=6,
    max_zoom=9,
).add_to(m)

TileLayer(
    tiles=nisar_track_frequencyA_tilejson["tiles"][0],
    name="NISAR GCOV frequencyA (higher res)",
    opacity=1,
    attr="NASA/ISRO",
    overlay=True,
    min_zoom=10,
).add_to(m)

LayerControl().add_to(m)

m

### Visualizing multiple orbits/tracks

It is possible to visualize a mosaic across multiple orbits/tracks, but this will result in TiTiler-CMR opening more granules per tile request and due to the structure of the granule files this is a resource intensive process! These visualizations should be used with a higher minimum zoom level than the orbit-wise tile layers.

In [ ]:
# write an attribute filter to select only granules on ascending orbits
ascending_params = {"attribute": "string,ASCENDING_DESCENDING,ASCENDING"}

nisar_ascending_frequencyA_tilejson_req = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params={
        **nisar_tilejson_params,
        **nisar_frequencyA_params,
        **ascending_params,
    },
    timeout=None,
)

nisar_ascending_frequencyA_tilejson_req.raise_for_status()
nisar_ascending_frequencyA_tilejson = nisar_ascending_frequencyA_tilejson_req.json()

nisar_ascending_frequencyB_tilejson_req = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params={
        **nisar_tilejson_params,
        **nisar_frequencyB_params,
        **ascending_params,
        # minzoom must be higher since TiTiler-CMR will struggle with too many of these granules!
        "minzoom": 8,
    },
    timeout=None,
)

nisar_ascending_frequencyB_tilejson_req.raise_for_status()
nisar_ascemdomg_frequencyB_tilejson = nisar_ascending_frequencyB_tilejson_req.json()

In [ ]:
m = Map(
    location=((india_bbox[1] + india_bbox[3]) / 2, (india_bbox[0] + india_bbox[2]) / 2),
    zoom_start=8,
)

nisar_ascending_features = {
    "type": "FeatureCollection",
    "features": [
        feature
        for feature in nisar_assets_geojson["features"]
        if feature["properties"]["ASCENDING_DESCENDING"] == "ASCENDING"
    ],
}

GeoJson(
    nisar_ascending_features,
    name="granule footprints",
    popup=GeoJsonPopup(
        fields=["granule_ur", "datetime", "ASCENDING_DESCENDING", "TRACK_NUMBER"]
    ),
    style_function=lambda x: {
        "color": "orange",
        "weight": 2,
        "fillOpacity": 0,
    },
).add_to(m)

TileLayer(
    tiles=nisar_ascemdomg_frequencyB_tilejson["tiles"][0],
    name="NISAR GCOV frequencyB (lower res)",
    opacity=1,
    attr="NASA/ISRO",
    overlay=True,
    min_zoom=8,
    max_zoom=10,
).add_to(m)

TileLayer(
    tiles=nisar_ascending_frequencyA_tilejson["tiles"][0],
    name="NISAR GCOV frequencyA (higher res)",
    opacity=1,
    attr="NASA/ISRO",
    overlay=True,
    min_zoom=11,
).add_to(m)

LayerControl().add_to(m)

m

## OPERA Surface Displacement from Sentinel-1

Some collections require very careful filtering in order to produce valid mosaics. The OPERA Surface Displacement from Sentinel-1 collection contains granules that represent a comparison between two specific points in time. For this reason it does not make much sense to create mosaics of granules where those points in time do not match very closely.

The granules describe the reference and secondary timestamps in the additional attributes `REFERENCE_ZERO_DOPPLER_START_TIME` and `SECONDARY_ZERO_DOPPLER_END_TIME`. These values are represented as strings in the metadata but the CMR API allows you to filter down to granules where the additional attribute value falls in a range. This works in this case because the string datetimes are ordered alphabetically/chronologically.

In [ ]:
opera_disp_collection_concept_id = "C3294057315-ASF"

ca_bbox = (-127, 31, -113, 42)

# we want granules that share the same start/end date in the interval attributes
start_datetime = datetime(2016, 7, 5, tzinfo=UTC)
end_datetime = datetime(2017, 1, 25, tzinfo=UTC)

reference_interval = (
    start_datetime,
    start_datetime + timedelta(days=1) - timedelta(seconds=1),
)

secondary_interval = (
    end_datetime,
    end_datetime + timedelta(days=1) - timedelta(seconds=1),
)

dt_fmt = "%Y-%m-%dT%H:%M:%SZ"

opera_disp_params = {
    "collection_concept_id": opera_disp_collection_concept_id,
    "attribute": [
        f"string,REFERENCE_ZERO_DOPPLER_START_TIME,{reference_interval[0].strftime(dt_fmt)},{reference_interval[1].strftime(dt_fmt)}",
        f"string,SECONDARY_ZERO_DOPPLER_END_TIME,{secondary_interval[0].strftime(dt_fmt)},{secondary_interval[1].strftime(dt_fmt)}",
    ],
    "sortkey": "-start_date",
}

opera_disp_assets_req = httpx.get(
    f"{titiler_endpoint}/bbox/{','.join(str(b) for b in ca_bbox)}/assets",
    params={
        "skipcovered": False,
        "limit": 500,
        "f": "geojson",
        **opera_disp_params,
    },
    timeout=None,
)

opera_disp_assets_req.raise_for_status()
opera_disp_assets_geojson = opera_disp_assets_req.json()
print("found", len(opera_disp_assets_geojson["features"]), "granules")

In [ ]:
for granule in opera_disp_assets_geojson["features"]:
    print(granule["properties"]["granule_ur"])

In [ ]:
print(json.dumps(granule["properties"]["additional_attributes"], indent=2))

We can request a tilejson that can be used to render tiles for this mosaic of two granules like this: 

In [ ]:
opera_disp_tilejson_req = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params={
        **opera_disp_params,
        "bounding_box": ",".join(str(b) for b in ca_bbox),
        "exitwhenfull": True,
        "variables": ["displacement"],
        "rescale": [[-0.05, 0.05]],
        "colormap_name": "rainbow",
        "minzoom": 6,
    },
    timeout=None,
)

opera_disp_tilejson_req.raise_for_status()
opera_disp_tilejson = opera_disp_tilejson_req.json()
print(opera_disp_tilejson)

In [ ]:
m = Map(
    location=((ca_bbox[1] + ca_bbox[3]) / 2, (ca_bbox[0] + ca_bbox[2]) / 2),
    zoom_start=7,
)

GeoJson(
    opera_disp_assets_geojson,
    name="granule footprints",
    popup=GeoJsonPopup(fields=["granule_ur"]),
    style_function=lambda x: {
        "color": "magenta",
        "weight": 2,
        "fillOpacity": 0,
    },
).add_to(m)

TileLayer(
    tiles=opera_disp_tilejson["tiles"][0],
    name="OPERA-DISP",
    opacity=1,
    attr="NASA",
    min_zoom=6,
).add_to(m)

LayerControl().add_to(m)


m